# Train THOR-30M on Google Colab

This notebook trains **THOR-30M** (31.1M parameters) from scratch on the TinyStories dataset, using THOR's own tokenizer, model and training code from this repository.

**Requirements**
- A free Colab GPU: *Runtime → Change runtime type → T4 GPU*. No paid services are used.
- About 3 GB of Google Drive space. Checkpoints and the prepared dataset are saved to `MyDrive/thor/`, so they survive disconnects.

**Resuming:** if Colab disconnects, reconnect and choose *Runtime → Run all*. Finished steps are skipped and training continues from the last checkpoint (saved every 250 steps). Stopping a cell also saves a checkpoint before exiting.

**Time (estimates, not measurements):** about 5–10 min to download, about 15–30 min to prepare the data once, and roughly 2–4 hours of training on a T4 for the full 5,000 steps (≈655M tokens). This may take more than one Colab session.

**Before you start:** check the license on the [TinyStories dataset card](https://huggingface.co/datasets/roneneldan/TinyStories) and record it in `models/DATA.md`.

In [ ]:
# Settings
import os

REPO_URL = "https://github.com/ArjunChat/thor"
DATA_URL = "https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-train.txt"
RUN_NAME = "thor-30m-tinystories"
SESSION_STEPS = 0  # >0: stop (with a checkpoint) after this many steps in this session

# Testing switch for maintainers: runs every cell on CPU with a tiny model and synthetic text.
SMOKE_TEST = os.environ.get("THOR_SMOKE_TEST") == "1"

In [ ]:
# 1. Check for a GPU
import torch

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    major, minor = torch.cuda.get_device_capability(0)
    precision = "bf16" if major >= 8 else "fp16 + gradient scaling"
    print(f"GPU: {name} (compute {major}.{minor}) -> training precision: {precision}")
elif SMOKE_TEST:
    print("Smoke test: running on CPU")
else:
    raise RuntimeError("No GPU found. Runtime -> Change runtime type -> T4 GPU, then run all cells again.")

In [ ]:
# 2. Storage: Google Drive for anything that must survive a disconnect
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/thor")  # persistent
    WORK = Path("/content/work")                # fast local disk, wiped on disconnect
except ImportError:  # not on Colab (smoke test / local run)
    ROOT = Path(os.environ.get("THOR_ROOT", "thor-colab")).resolve()
    WORK = ROOT / "work"

DRIVE_DATA = ROOT / "data" / "tinystories"
LOCAL_DATA = WORK / "data" / "tinystories"
RUN_DIR = ROOT / "runs" / RUN_NAME
for p in (ROOT, WORK, RUN_DIR.parent):
    p.mkdir(parents=True, exist_ok=True)
print("Persistent storage:", ROOT)

In [ ]:
# 3. Get the THOR code
import signal
import subprocess
import sys

CODE = Path(os.environ["THOR_CODE_DIR"]) if os.environ.get("THOR_CODE_DIR") else WORK / "thor"
if not (CODE / ".git").exists():
    url = REPO_URL
    try:  # Private repo? Add a GitHub token as a Colab secret named GITHUB_TOKEN.
        from google.colab import userdata
        url = REPO_URL.replace("https://", f"https://x-access-token:{userdata.get('GITHUB_TOKEN')}@")
    except Exception:
        pass
    subprocess.run(["git", "clone", "--depth", "1", url, str(CODE)], check=True)
elif not os.environ.get("THOR_CODE_DIR"):
    subprocess.run(["git", "-C", str(CODE), "pull", "--ff-only"], check=True)
print(subprocess.run(["git", "-C", str(CODE), "log", "-1", "--format=THOR code at %h: %s"],
                     capture_output=True, text=True).stdout)


def thor(*args):
    # Run `python -m thor.<...>` from models/, streaming output. Stopping the cell
    # forwards Ctrl-C so training saves a checkpoint before it exits.
    cmd = [sys.executable, "-m", *map(str, args)]
    print("$", " ".join(cmd[2:]))
    proc = subprocess.Popen(cmd, cwd=CODE / "models", stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in proc.stdout:
            print(line, end="")
    except KeyboardInterrupt:
        proc.send_signal(signal.SIGINT)
        for line in proc.stdout:
            print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}")


thor("thor.specs")

In [ ]:
# 4. Prepare data once: download, train the tokenizer, tokenize. Saved to Drive.
import shutil
import urllib.request

FILES = ["tokenizer.json", "train.bin", "val.bin", "meta.json"]  # meta.json last = complete


def copy_dataset(src, dst):
    dst.mkdir(parents=True, exist_ok=True)
    for name in FILES:
        shutil.copy2(src / name, dst / name)


if not (DRIVE_DATA / "meta.json").exists():
    raw = WORK / "raw" / "tinystories.txt"
    raw.parent.mkdir(parents=True, exist_ok=True)
    if SMOKE_TEST:
        stories = [f"Once upon a time, a little robot named Bit found {i} shiny stars. "
                   f"Bit was happy and shared them with a friend. The end." for i in range(400)]
        raw.write_text("\n<|endoftext|>\n".join(stories), encoding="utf-8")
    elif not raw.exists():
        print("Downloading", DATA_URL)
        tmp = raw.with_suffix(".part")
        urllib.request.urlretrieve(DATA_URL, tmp)
        tmp.rename(raw)
    print(f"Raw text: {raw.stat().st_size / 1e9:.2f} GB")
    thor("thor.prepare", "--input", raw, "--out", LOCAL_DATA, "--overwrite",
         "--vocab-size", 512 if SMOKE_TEST else 16384, "--doc-sep", "<|endoftext|>",
         "--val-fraction", 0.05 if SMOKE_TEST else 0.005, "--workers", os.cpu_count() or 1)
    copy_dataset(LOCAL_DATA, DRIVE_DATA)
    print("Saved prepared data to", DRIVE_DATA)
elif not (LOCAL_DATA / "meta.json").exists():
    copy_dataset(DRIVE_DATA, LOCAL_DATA)
    print("Copied prepared data from Drive to local disk")

print((LOCAL_DATA / "meta.json").read_text())

In [ ]:
# 5. Train (resumes automatically from RUN_DIR/ckpt_last.pt)
args = ["thor.train", "--config", "configs/thor-30m-tinystories.json",
        "--data-dir", LOCAL_DATA, "--out-dir", RUN_DIR,
        # T4 memory: micro-batch 16 x 8 accumulation = same 131k tokens/step as the config.
        "--batch-size", 16, "--grad-accum", 8]
if SESSION_STEPS:
    args += ["--session-steps", SESSION_STEPS]
if SMOKE_TEST:
    args += ["--preset", "thor-nano", "--seq-len", 64, "--batch-size", 4, "--grad-accum", 1,
             "--max-steps", 40, "--warmup-steps", 5, "--eval-interval", 20, "--eval-iters", 2,
             "--ckpt-interval", 20, "--log-interval", 10, "--keep-every", 0, "--device", "cpu"]
thor(*args)

In [ ]:
# 6. Loss curves
import json

records = [json.loads(l) for l in (RUN_DIR / "metrics.jsonl").read_text().splitlines() if l.strip()]
train_pts = [(r["step"], r["loss"]) for r in records if "loss" in r]
val_pts = [(r["step"], r["eval_val_loss"]) for r in records if "eval_val_loss" in r]
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(8, 4))
    plt.plot(*zip(*train_pts), label="train (per step)", alpha=0.6)
    if val_pts:
        plt.plot(*zip(*val_pts), "o-", label="validation")
    plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.grid(alpha=0.3)
    plt.title(RUN_NAME); plt.show()
except ImportError:
    pass
print("latest:", train_pts[-1], "| best validation:", min(val_pts, key=lambda p: p[1]) if val_pts else None)

In [ ]:
# 7. Evaluate the best checkpoint on held-out text
thor("thor.evaluate", "--ckpt", RUN_DIR / "model_best.pt", "--data-dir", LOCAL_DATA,
     "--device", "cuda" if torch.cuda.is_available() else "cpu",
     *(["--max-tokens", 20000] if SMOKE_TEST else []))

In [ ]:
# 8. Generate samples
for prompt in ["Once upon a time", "The little dog was sad because", "Lily and Ben went to the park."]:
    thor("thor.sample", "--ckpt", RUN_DIR / "model_best.pt", "--tokenizer", LOCAL_DATA / "tokenizer.json",
         "--prompt", prompt, "--max-new-tokens", 20 if SMOKE_TEST else 150, "--temperature", 0.8,
         "--top-k", 50, "--device", "cuda" if torch.cuda.is_available() else "cpu")

## After training

1. **Record results exactly as measured.** Copy the `thor.evaluate` output and final `metrics.jsonl` values into a model card made from `docs/model-cards/TEMPLATE.md`. Don't round up or estimate.
2. **Update `models/DATA.md`** with the dataset file, download date, license and token counts (from `meta.json`).
3. **Update the status:** THOR-30M becomes *Trained (internal)*. *Released* is a separate decision (weights hosting, license).
4. Checkpoints live in `MyDrive/thor/runs/thor-30m-tinystories/`. `model_best.pt` holds the weights only (~125 MB); `ckpt_last.pt` also holds optimizer state for resuming.

**What to expect:** a 30M-parameter model trained on TinyStories writes simple, mostly grammatical children's stories with frequent logic slips. It is a research and education model, not a general assistant.